In [1]:
!pip install -U firecrawl-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [firecrawl-py]━━━━━ 2/3 [firecrawl-py]


In [2]:
import os
os.environ["FIRECRAWL_API_KEY"] = "fc-7cd82eef0d4c4cd9b069dfa85f8ca741"  

In [3]:
from firecrawl import Firecrawl
app = Firecrawl(api_key=os.environ["FIRECRAWL_API_KEY"])


In [4]:
TEST_URL = "https://www.g2.com/products/asana/reviews"

r_true = app.scrape(TEST_URL, formats=["markdown"], only_main_content=True, max_age=0)
r_false = app.scrape(TEST_URL, formats=["markdown"], only_main_content=False, max_age=0)

print(f"only_main_content=True:  {len(r_true.markdown)} chars")
print(f"only_main_content=False: {len(r_false.markdown)} chars")
print(f"Expected True <= False: {len(r_true.markdown) <= len(r_false.markdown)}")

only_main_content=True:  19157 chars
only_main_content=False: 57520 chars
Expected True <= False: True


In [5]:
# ============================================================
# Cell 4 — Quality gate, same shape as the Crawl4AI version
# ============================================================
MIN_LEN = 200

def select_markdown(result) -> tuple[str | None, str]:
    text = result.markdown
    if text and len(text) >= MIN_LEN:
        return text, "ok"
    return None, "failed"

text, tier = select_markdown(r_true)
print(f"\nquality_tier: {tier}, length: {len(text) if text else 0}")


quality_tier: ok, length: 19157


In [ ]:
# ============================================================
# Cell 5 — Your full 15-URL corpus, single extraction path
# ============================================================
documents = [
    {"url": "https://www.g2.com/products/asana/reviews", "brand": "Asana"},
    {"url": "https://www.g2.com/products/trello/reviews", "brand": "Trello"},
    {"url": "https://www.g2.com/compare/asana-vs-trello", "brand": "Asana"},
    {"url": "https://www.g2.com/compare/asana-vs-monday-com", "brand": "Asana"},
    {"url": "https://learn.g2.com/best-project-management-software", "brand": "ClickUp"},
    {"url": "https://www.capterra.com/project-management-software/s/free/", "brand": "Trello"},
    {"url": "https://softwarefinder.com/resources/trello-vs-asana-vs-monday-vs-clickup", "brand": "Trello"},
    {"url": "https://www.probackup.io/blog/asana-vs-trello-vs-monday-com-which-project-management-platform-is-right-for-you", "brand": "Asana"},
    {"url": "https://saascompared.io/blog/best-project-management-software-2026/", "brand": "ClickUp"},
    {"url": "https://till-freitag.com/en/blog/best-project-management-tools", "brand": "monday.com"},
    {"url": "https://www.techradar.com/best/best-project-management-software", "brand": "Asana"},
    {"url": "https://thedigitalprojectmanager.com/tools/best-project-management-software/", "brand": "monday.com"},
    {"url": "https://zapier.com/blog/free-project-management-software/", "brand": "Trello"},
    {"url": "https://zapier.com/blog/task-management-software/", "brand": "Asana"},
    {"url": "https://asana.com/resources/best-project-management-software", "brand": "Asana"},
]

results = []
for doc in documents:
    try:
        r = app.scrape(doc["url"], formats=["markdown"], only_main_content=True, max_age=0)
        text, tier = select_markdown(r)
        results.append({
            "url": doc["url"], "brand": doc["brand"],
            "length": len(text) if text else 0, "tier": tier,
        })
        print(f"[{tier:>6}] {doc['brand']:<12} {len(text) if text else 0:>6} chars  {doc['url']}")
    except Exception as e:
        results.append({"url": doc["url"], "brand": doc["brand"], "error": str(e), "tier": "error"})
        print(f"[ERROR] {doc['brand']:<12} {doc['url']} — {e}")


In [ ]:
# ============================================================
# Cell 6 — Inspect one result in full, to eyeball residual noise
# ============================================================
print(r_true.markdown)  # scroll through the whole thing, not just a preview

In [ ]:
result = app.scrape(
    doc["url"],
    formats=["markdown"],
    only_main_content=True,
    exclude_tags=["img", "svg", "picture", "video"],
    max_age=0,
)

In [ ]:
from firecrawl import Firecrawl

app = Firecrawl(api_key=os.environ["FIRECRAWL_API_KEY"])

TEST_URL = "https://www.g2.com/products/asana/reviews"
scraped = app.scrape(
    TEST_URL, formats=["markdown"], only_main_content=True,
    exclude_tags=["img", "svg", "picture", "video"], max_age=0,
)
print(f"Extracted {len(scraped.markdown)} chars")

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import tiktoken, hashlib

encoding = tiktoken.get_encoding("cl100k_base")
def token_len(text: str) -> int:
    return len(encoding.encode(text))

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3"), ("####", "h4")],
    strip_headers=False,
)
chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50, length_function=token_len,
)

def chunk_document(markdown_text: str, doc_id: str, tenant_id: str, brand: str, source: str) -> list[Document]:
    sections = header_splitter.split_text(markdown_text)
    docs, idx = [], 0
    for section in sections:
        heading_path = " > ".join(section.metadata.values()) or "root"
        for piece in chunk_splitter.split_text(section.page_content):
            docs.append(Document(
                page_content=piece,
                metadata={
                    "tenant_id": tenant_id, "doc_id": doc_id, "chunk_index": idx,
                    "source": source, "brand": brand, "heading_path": heading_path,
                    "token_count": token_len(piece),
                },
            ))
            idx += 1
    return docs

In [ ]:
doc_id = hashlib.sha256(TEST_URL.encode()).hexdigest()
chunks = chunk_document(scraped.markdown, doc_id=doc_id, tenant_id="acme", brand="Asana", source=TEST_URL)

print(f"Total chunks: {len(chunks)}")
for i, c in enumerate(chunks[:5]):
    print(f"\n--- chunk {i} | heading: {c.metadata['heading_path']} | tokens: {c.metadata['token_count']} ---")
    print(c.page_content)

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="brand_chunks",
    vectors_config={"dense": models.VectorParams(size=1536, distance=models.Distance.COSINE)},
    sparse_vectors_config={"sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)},
)
client.create_payload_index(
    collection_name="brand_chunks", field_name="metadata.tenant_id",
    field_schema=models.PayloadSchemaType.KEYWORD,
)
print("Collection created")